In [1]:
import pandas as pd 
import numpy as np

In [2]:
data = pd.read_csv('../data/labeled_articles.csv', index_col='id')
data

,Unnamed: 0,text,authenticity_score,sensationalism_score,political_bias_score,spam_score,confirmation_bias_score,short_term_utility_score
id,,,,,,,,
1,0,"NEW YORK, NY — Mayor-elect Zohran Mamdani anno...",2,7,5,9,3,8
2,1,Walmart is proving to be America’s antidote to...,7,5,2,1,5,3
3,2,LAS VEGAS—Shaking his head in frustration afte...,1,9,2,5,8,10
4,3,Nov 20 (Reuters) - Studies from Novo Nordisk (...,9,4,1,2,6,2
5,4,The Buffalo Bills and Houston Texans will meet...,8,2,1,1,6,5
6,5,"For the first time in his second term, Preside...",7,6,8,4,2,6
7,6,While hosting Saudi Crown Prince Mohammed bin ...,9,7,1,8,5,8
8,7,SAN DIEGO — San Diego homeowners looking to se...,7,2,1,1,6,4
9,8,The Democrat is accused of stealing Federal Em...,8,7,7,6,8,3


# Function and Prompt Definitions

In [3]:
import os
from openai import OpenAI
import json
import re
import yaml

with open("../config.yaml", "r") as f:
    config = yaml.safe_load(f)

api_key = config["nautilus"]["api"]

In [4]:
def baseline_prompt(article_text, pred_vec=None):
    return f"""
You are an AI tasked with scoring news articles. 

For the following article, assign numeric scores from 1 to 10 for each of these six factors: 

1. Authenticity
2. Sensationalism
3. Political Bias
4. Spam
5. Confirmation Bias
6. Short-Term Utility

Do not provide explanations, examples, or comments. Only output a Python dictionary with these exact keys and numeric values.  

Article:
{article_text}
"""

def refined_prompt(article_text, pred_vec=None):
    return f"""
You are an AI assistant assigned to evaluate the factuality of news statements using a generative fact-checking pipeline.
Your task is to analyze article text, incorporate predictive model outputs WITHOUT overweighting them, and compute factor scores using the scoring recipes below.

VECTOR DESCRIPTION
The feature vector contains the following predictive model outputs and auxiliary measures:
0-5: Probabilities for truthfulness classes from our custom BERT-based model:
     0 = False, 1 = Half True, 2 = Mostly True, 3 = True, 4 = Barely True, 5 = Pants on Fire
7: Count of numeric/statistical entities detected in the text
8: Count of conservative bigram matches in the text
9: Count of liberal bigram matches in the text
10: Emotional intensity score (absolute VADER compound score)
11: Spam likelihood score (0–1, probability of being spam)

ANTI-BIAS CONSTRAINT
- Treat predictive model scores only as auxiliary context.
- Do NOT give these predictive scores undue weight.
- If your analysis of the article text contradicts the predictive scores, rely on the TEXTUAL EVIDENCE and explain the discrepancy.

==========================
FACTUALITY FACTORS (6 TOTAL)
==========================

1. AUTHENTICITY
- Definition: Does the text present evidence that the claims are genuine, verifiable, and traceable?
- Scoring Recipe (1–10): Look for verifiable details, named sources, timestamps, data, official statements; higher when concrete and falsifiable.
- Output: numeric score + 1–2 sentences referencing evidence.

2. SENSATIONALISM
- Definition: Presence of hyperbole, emotional language, exaggeration.
- Scoring Recipe (1–10): Extract emotional/hyperbolic language, count dramatic constructions, score based on density and prominence.
- Output: score + 2 example phrases.

3. POLITICAL BIAS
- Definition: Degree to which the article leans left, center, or right.
- Scoring Recipe (0–10 + tag): Identify partisan framing or selective omission.
- Output: numeric score + category {{left, centrist, right, mixed}} + examples.

4. Spam
- Definition: Determine whether a piece of content qualifies as spam, and assess whether the spam contains or contributes to disinformation.
- Scoring Recipe (1–10): Score based on how strongly the content exhibits spam characteristics.
- Output: score + example phrase.

5. CONFIRMATION BIAS
- Definition: Selective presentation of information reinforcing a preferred conclusion.
- Scoring Recipe (1–10): Identify cherry-picked evidence or missing counterarguments.
- Output: score + 1 example.

6. SHORT-TERM UTILITY (Profit Incentive)
- Definition: Degree content maximizes clicks or engagement.
- Scoring Recipe (1–10): Detect clickbait, urgent calls to action, monetization cues.
- Output: score + 1–2 indicators of profit-driven framing.

OUTPUT FORMAT (STRICT JSON)
{{
  "veracity_label": "One of: True, Mostly True, Half True, Mostly False, False, Pants on Fire",
  "explanation_text": "A well-detailed explanation explaining the final verdict and reconciling any discrepancies. Explain each factuality factor's score choice as well",
  "factor_scores": [
    {{"factor": "Authenticity", "score": 1-10, "reasoning": "Brief evidence"}},
    {{"factor": "Sensationalism", "score": 1-10, "reasoning": "Brief evidence"}},
    {{"factor": "Political Bias", "score": 1-10, "reasoning": "Brief evidence"}},
    {{"factor": "Spam", "score": 1-10, "reasoning": "Brief evidence"}},
    {{"factor": "Confirmation Bias", "score": 1-10, "reasoning": "Brief evidence"}},
    {{"factor": "Short-term Utility", "score": 1-10, "reasoning": "Brief evidence"}}
  ]
}}

ARTICLE TEXT:
\"\"\"
{article_text}
\"\"\"

PREDICTIVE MODEL FEATURE VECTOR:
[{pred_vec}]
"""

def final_prompt(article_text, pred_vec):
    return f"""You are an AI assistant assigned to evaluate the factuality of news statements using a generative fact-checking pipeline.

Your task is to analyze article text, incorporate predictive model outputs WITHOUT overweighting them, and compute factor scores using the scoring recipes below.

ANTI-BIAS CONSTRAINT
- Treat predictive model scores only as auxiliary context.
- Do NOT give these predictive scores undue weight.
- If your analysis of the article text contradicts the predictive scores, rely on the TEXTUAL EVIDENCE and explain the discrepancy.

FACTUALITY FACTORS (6 TOTAL):

1. AUTHENTICITY
- Definition: Does the text present evidence that the claims are genuine, verifiable, and traceable?
- Scoring Recipe (1–10): Look for verifiable details, named sources, timestamps, data, official statements; higher when concrete and falsifiable.
- Output: numeric score + 1–2 sentences referencing evidence.

2. SENSATIONALISM
- Definition: Presence of hyperbole, emotional language, exaggeration.
- Scoring Recipe (1–10): Extract emotional/hyperbolic language, count dramatic constructions, score based on density and prominence.
- Output: score + 2 example phrases.

3. POLITICAL BIAS
- Definition: Degree to which the article leans left, center, or right.
- Scoring Recipe (0–10 + tag): Identify partisan framing or selective omission.
- Output: numeric score + category {{left, centrist, right, mixed}} + examples.

4. Spam
- Definition: Determine whether a piece of content qualifies as spam, and assess whether the spam contains or contributes to disinformation.
- Scoring Recipe (1–10): Score based on how strongly the content exhibits spam characteristics.
- Output: score + example phrase.

5. CONFIRMATION BIAS
- Definition: Selective presentation of information reinforcing a preferred conclusion.
- Scoring Recipe (1–10): Identify cherry-picked evidence or missing counterarguments.
- Output: score + 1 example.

6. SHORT-TERM UTILITY (Profit Incentive)
- Definition: Degree content maximizes clicks or engagement.
- Scoring Recipe (1–10): Detect clickbait, urgent calls to action, monetization cues.
- Output: score + 1–2 indicators of profit-driven framing.

VERACITY LABEL (REQUIRED)
- Output a final factuality classification: pants-fire, false, barely-true, half-true, mostly-true, true

OUTPUT FORMAT (STRICT JSON)
{{
  "veracity_label": "One of: True, Mostly True, Half True, Mostly False, False, Pants on Fire",
  "explanation_text": "A well-detailed explanation explaining the final verdict and reconciling any discrepancies. Explain each factuality factor's score choice as well",
  "factor_scores": [
    {{"factor": "Authenticity", "score": 1-10, "reasoning": "Brief evidence"}},
    {{"factor": "Sensationalism", "score": 1-10, "reasoning": "Brief evidence"}},
    {{"factor": "Political Bias", "score": 1-10, "reasoning": "Brief evidence"}},
    {{"factor": "Spam", "score": 1-10, "reasoning": "Brief evidence"}},
    {{"factor": "Confirmation Bias", "score": 1-10, "reasoning": "Brief evidence"}},
    {{"factor": "Short-term Utility", "score": 1-10, "reasoning": "Brief evidence"}}
  ]
}}

REASONING FORMAT
- Provide concise, structured rationale per factor: key textual evidence, weighting of predictive scores, numeric reasoning.  
- DO NOT reveal internal chain-of-thought or hidden reasoning.

EXAMPLES
Here are some examples of the expected output format and reasoning structure:
    - Example 1:
    input: 
    {{
        "article": "A new study shows that drinking green tea daily reduces the risk of heart disease by 30%. The study surveyed 10,000 adults over 5 years and was published in the Journal of Cardiology.", 
        "source": "Health Daily News"
    }}
    result:
    {{
        "veracity_label": "Mostly True",
        "explanation_text": "The article cites a study published in a reputable journal with a large sample size and multi-year scope, which strongly supports its credibility. While the claim that green tea reduces heart disease risk by 30% is plausible, the phrasing slightly overstates certainty by not discussing limitations or confidence intervals. There is no political agenda or hostile language, and the article is primarily informational rather than engagement-driven.",
        "factor_scores": [
            {{"factor": "Authenticity", "score": 9, "reasoning": "Published in a reputable journal with a large sample size and clear methodology."}},
            {{"factor": "Sensationalism", "score": 3, "reasoning": "Slightly bold phrasing but largely factual and restrained."}},
            {{"factor": "Political Bias", "score": 1, "reasoning": "No political framing or agenda detected."}},
            {{"factor": "Spam", "score": 1, "reasoning": "No deceptive, repetitive, or manipulative content."}},
            {{"factor": "Confirmation Bias", "score": 2, "reasoning": "Emphasizes positive findings with limited discussion of caveats."}},
            {{"factor": "Short-term Utility", "score": 2, "reasoning": "Primarily informational with mild attention-grabbing appeal."}}
        ]
    }}
    - Example 2:
    input:
    {{
        "article": "Politician X is the worst leader in history! Everything they touch fails, and the economy is collapsing under their rule.", 
        "source": "Partisan Weekly"}}
    result:
    {{
        "veracity_label": "False",
        "explanation_text": "The article makes sweeping negative claims about a political figure without citing evidence or verifiable data. The language is highly emotional, attacking the individual rather than evaluating specific policies or outcomes. Strong political bias and selective presentation of information undermine credibility, suggesting the content is designed more to provoke outrage than inform.",
        "factor_scores": [
            {{"factor": "Authenticity", "score": 2, "reasoning": "Claims are vague and unsupported by evidence or sources."}},
            {{"factor": "Sensationalism", "score": 9, "reasoning": "Uses hyperbolic and extreme language to provoke emotion."}},
            {{"factor": "Political Bias", "score": 10, "reasoning": "Explicit partisan framing targeting a political figure."}},
            {{"factor": "Spam", "score": 8, "reasoning": "Aggressive rhetoric and personal attacks resemble engagement-driven spam content."}},
            {{"factor": "Confirmation Bias", "score": 8, "reasoning": "Only negative information is presented; counterpoints are ignored."}},
            {{"factor": "Short-term Utility", "score": 7, "reasoning": "Clearly designed to trigger outrage and maximize clicks."}}
        ]
    }}
    - Example 3:
    input:
    {{
        "article": "Local bakery wins award for best chocolate cake. The contest included 50 bakeries, and judges highlighted creativity and flavor balance.", 
        "source": "Town Gazette"
    }}
    result:
    {{
        "veracity_label": "True",
        "explanation_text": "The article reports on a verifiable local contest with identifiable participants and judging criteria. The tone is neutral and factual, with no exaggeration or bias. The content is a straightforward human-interest story with no indication of manipulation or selective framing.",
        "factor_scores": [
            {{"factor": "Authenticity", "score": 8, "reasoning": "Contest details and judging process are verifiable and clearly described."}},
            {{"factor": "Sensationalism", "score": 2, "reasoning": "Positive tone without exaggeration or emotional manipulation."}},
            {{"factor": "Political Bias", "score": 1, "reasoning": "No political content or implications."}},
            {{"factor": "Spam", "score": 1, "reasoning": "No misleading or manipulative characteristics."}},
            {{"factor": "Confirmation Bias", "score": 1, "reasoning": "Balanced reporting with no selective omission."}},
            {{"factor": "Short-term Utility", "score": 3, "reasoning": "Lightly engaging but not engineered for virality."}}
        ]
    }}

ARTICLE TEXT:
[{article_text}]

PREDICTED VECTOR:
[{pred_vec}]
"""

def fcot_prompt4(text, feature_vector=None):
    print(text)
    prompt = f"""
You are an expert Fact-Checking AI. Your objective is to evaluate news veracity by synthesizing raw text with predictive model outputs through a recursive, self-correcting reasoning architecture.

==========================
I. RECURSIVE REASONING ARCHITECTURE (INTERNAL)
==========================
Before generating the JSON, execute three layers of reasoning. Each layer must maximize a specific objective while minimizing a specific error.

LAYER 1: LOCAL SIGNAL EXTRACTION
- Objective: Maximize detection of specific textual markers (entities, quotes, emotional triggers).
- Minimization: Minimize "Surface Bias" (ignoring subtle nuances or sarcasm).
- Aperture: Narrow. Focus on local grammar, syntax, and specific claims.

LAYER 2: OBJECTIVE RECONCILIATION & SELF-CORRECTION
- Objective: Maximize coherence between the provided Model Vector and your Layer 1 findings.
- Minimization: Minimize "Model Anchoring." If the BERT model says "Pants on Fire" but the text is a dry, sourced report, you MUST aggressively correct the model's error using textual evidence.
- Aperture: Mid-range. Compare the article's internal logic against the statistical probabilities provided in the Vector.

LAYER 3: GLOBAL CONTEXT & EPISTEMIC DIVERSITY
- Objective: Maximize novelty of perspective by considering "why" and "for whom" the content was written.
- Minimization: Minimize "Echo Chamber" reasoning. Explicitly look for what is MISSING (omitted counter-arguments or hidden incentives).
- Aperture: Wide. Integrate sociopolitical trends, profit motives, and global priors to finalize the "Short-term Utility" and "Bias" scores.

==========================
II. DATA INPUTS
==========================
PREDICTIVE MODEL VECTOR:
- 0-5: BERT Probabilities (0:False, 1:Half True, 2:Mostly True, 3:True, 4:Barely True, 5:Pants on Fire)
- 7: Numeric/Statistical count | 8: Conservative bigrams | 9: Liberal bigrams
- 10: Emotional Intensity (VADER) | 11: Spam Likelihood (0-1)

ARTICLE TEXT:
{text}

==========================
III. FACTUALITY FACTORS (SCORING RECIPES)
==========================
1. AUTHENTICITY (1–10): Score based on traceability. Are sources named or anonymous? Is there a timestamped paper trail?
2. SENSATIONALISM (1–10): Density of hyperbole. (e.g., "Shocking," "Destroyed," "Outrage").
3. POLITICAL BIAS (0–10 + Tag): Identify partisan framing {left, centrist, right, mixed}.
4. SPAM (1–10): Degree of content farm characteristics or disinformation contribution.
5. CONFIRMATION BIAS (1–10): Does it cherry-pick data to feed a specific narrative?
6. SHORT-TERM UTILITY (1–10): Detect monetization cues, clickbait, and engagement-hacking.

==========================
IV. OUTPUT CONSTRAINTS
==========================
- Perform all recursive self-correction internally.
- OUTPUT ONLY the final JSON object.
- The "explanation_text" must explicitly mention how you reconciled any conflicts between the Model Vector and the Text.

{{
    "veracity_label": "String",
    "explanation_text": "Detailed synthesis of layers 1-3",
    "factor_scores": [
        {"factor": "Name", "score": Int, "reasoning": "Evidence-based justification"}
    ]
}}
"""
    return prompt

def fcot_prompt3_gpt(text,feature_vector=None):
    prompt =  f""" 
You are an AI assistant assigned to evaluate the factuality of news statements using a generative fact-checking pipeline. Your task is to analyze article text, incorporate predictive model outputs WITHOUT overweighting them, and compute factor scores using the scoring recipes below. 

==========================
INTERNAL MULTI-PASS REASONING (DO NOT REVEAL)
==========================
Use a fractal, multi-layer reasoning process internally:

Layer 1 — Surface Scan: detect claims, entities, sources; maximize signal, minimize missed red flags
Layer 2 — Evidence Decomposition: verify claims; maximize verifiability, minimize assumptions
Layer 3 — Factor Micro-Analysis: score each factor; maximize textual alignment, minimize cross-factor bias
Layer 4 — Model vs Text Reconciliation: compare predictive outputs; maximize textual evidence priority, minimize model over-weighting
Layer 5 — Consistency & Calibration: ensure coherence; maximize score consistency, minimize contradictions
Layer 6 — Recursive Self-Critique: revise any weak or inconsistent scoring; maximize justification, minimize unsupported assertions
Aperture Expansion: gradually integrate external context and historical patterns in Layers 2–4

PASS 1 — Surface Scan:
- Identify headline claims, entities, tone, and source cues.
- Note any immediate red flags (satire, parody, lack of sourcing, emotional framing).

PASS 2 — Evidence Decomposition:
- Initially analyze claims based on the article alone
- Later, incorporate broader context:
    - Known source reliability
    - Historical claim patterns
    - Publicly available reference data

PASS 3 — Factor-Specific Micro-Analysis:
For EACH factuality factor:
- Re-evaluate the text specifically for that factor.
- Extract concrete textual signals (phrases, omissions, framing).
- Independently score based on the scoring recipe (do not anchor to other factors).

PASS 4 — Model vs Text Reconciliation:
- Compare predictive model outputs with your textual analysis.
- If they disagree, prioritize textual evidence.
- Explicitly reconcile contradictions in the explanation_text.

PASS 5 — Consistency & Calibration Check:
- Maximize internal consistency across factor scores
- Minimize discrepancies with textual evidence
- If inconsistencies remain, recursively revisit Layer 2-4 to refine scores

PASS 6 — Self-Critique:
- Maximize justification for each factor score
- Minimize reliance on assumptions or predictive model bias
- Revise any factor lacking sufficient textual grounding

IMPORTANT:
- Perform all reasoning internally.
- DO NOT reveal chain-of-thought, step-by-step reasoning, or internal notes.
- ONLY output the final strict JSON object.

VECTOR DESCRIPTION 
The feature vector contains the following predictive model outputs and auxiliary measures: 
0-5: Probabilities for truthfulness classes from our custom BERT-based model: 
    0 = False, 1 = Half True, 2 = Mostly True, 3 = True, 4 = Barely True, 5 = Pants on Fire 
    7: Count of numeric/statistical entities detected in the text 
    8: Count of conservative bigram matches in the text 
    9: Count of liberal bigram matches in the text
    10: Emotional intensity score (absolute VADER compound score) 
    11: Spam likelihood score (0–1, probability of being spam) 

ANTI-BIAS CONSTRAINT 
- Treat predictive model scores only as auxiliary context. 
- Do NOT give these predictive scores undue weight. 
- If your analysis of the article text contradicts the predictive scores, rely on the TEXTUAL EVIDENCE and explain the discrepancy. 

========================== 
FACTUALITY FACTORS (6 TOTAL) 
========================== 
1. AUTHENTICITY 
- Definition: Does the text present evidence that the claims are genuine, verifiable, and traceable? 
- Scoring Recipe (1–10): Look for verifiable details, named sources, timestamps, data, official statements; higher when concrete and falsifiable. 
- Output: numeric score + 1–2 sentences referencing evidence. 

2. SENSATIONALISM 
- Definition: Presence of hyperbole, emotional language, exaggeration. 
- Scoring Recipe (1–10): Extract emotional/hyperbolic language, count dramatic constructions, score based on density and prominence. 
- Output: score + 2 example phrases. 

3. POLITICAL BIAS 
- Definition: Degree to which the article leans left, center, or right. 
- Scoring Recipe (0–10 + tag): Identify partisan framing or selective omission. 
- Output: numeric score + category {{left, centrist, right, mixed}} + examples. 

4. Spam 
- Definition: Determine whether a piece of content qualifies as spam, and assess whether the spam contains or contributes to disinformation. 
- Scoring Recipe (1–10): Score based on how strongly the content exhibits spam characteristics. 
- Output: score + example phrase. 

5. CONFIRMATION BIAS 
- Definition: Selective presentation of information reinforcing a preferred conclusion. 
- Scoring Recipe (1–10): Identify cherry-picked evidence or missing counterarguments. 
- Output: score + 1 example. 

6. SHORT-TERM UTILITY (Profit Incentive) 
- Definition: Degree content maximizes clicks or engagement. 
- Scoring Recipe (1–10): Detect clickbait, urgent calls to action, monetization cues. 
- Output: score + 1–2 indicators of profit-driven framing. 

SELF-CRITIQUE (INTERNAL):
- Before finalizing, challenge each score:
  "If I had to defend this score to a skeptic, is my evidence sufficient?"
- Revise any score that lacks strong textual grounding

OUTPUT FORMAT (STRICT JSON) 
{{ 
    "veracity_label": "One of: True, Mostly True, Half True, Mostly False, False, Pants on Fire", 
    "explanation_text": "A well-detailed explanation explaining the final verdict and reconciling any discrepancies. Explain each factuality factor's score choice as well", 
    "factor_scores": [ 
    {{"factor": "Authenticity", "score": 1-10, "reasoning": "Brief evidence"}}, 
    {{"factor": "Sensationalism", "score": 1-10, "reasoning": "Brief evidence"}}, 
    {{"factor": "Political Bias", "score": 1-10, "reasoning": "Brief evidence"}}, 
    {{"factor": "Spam", "score": 1-10, "reasoning": "Brief evidence"}}, 
    {{"factor": "Confirmation Bias", "score": 1-10, "reasoning": "Brief evidence"}}, 
    {{"factor": "Short-term Utility", "score": 1-10, "reasoning": "Brief evidence"}} 
    ] 
}} 
ARTICLE TEXT: 
\"\"\" {text} \"\"\" 
"""
    return prompt

def run_genai(article_text, feature_vector=None, prompt=None):
    prompt = prompt(article_text, feature_vector)
    client = OpenAI(
        api_key = api_key, 
        base_url = "https://ellm.nrp-nautilus.io/v1"
    )

    completion = client.chat.completions.create(
        model="gemma3",
        messages=[
          
            {"role": "system", "content": "You are a helpful assistant that outputs STRICT JSON only."},
            {"role": "user", "content": prompt},
        ],
        temperature=0.1 
    )
    return completion.choices[0].message.content

# Prompt evaluation

## Naive Prompt

In [5]:
responses_naive = []
for i in range(1,data.shape[0]+1):
    txt = data.loc[i, 'text']
    response = run_genai(article_text=txt, prompt=baseline_prompt)
    clean_str = re.sub(r"```json|```", "", response).strip()
    if "{" in clean_str:
        start = clean_str.find("{")
        end = clean_str.rfind("}") + 1
        clean_str = clean_str[start:end]
    response_json = json.loads(clean_str)
    responses_naive.append(response_json)

In [6]:
naive_df = pd.DataFrame(responses_naive)
naive_df

,Authenticity,Sensationalism,Political Bias,Spam,Confirmation Bias,Short-Term Utility
0,1,9,7,2,4,3
1,8,3,4,1,5,7
2,8,6,1,1,2,4
3,9,3,2,1,4,7
4,10,2,1,1,3,8
5,8,4,6,1,5,7
6,9,7,8,1,6,8
7,9,2,3,1,4,8
8,9,5,6,1,5,8
9,8,4,6,1,7,7


## Refined Prompt: Adding scoring recipes

In [7]:
responses_refined = []
for i in range(1,data.shape[0]+1):
    txt = data.loc[i, 'text']
    response = run_genai(article_text=txt, prompt=refined_prompt)
    clean_str = re.sub(r"```json|```", "", response).strip()
    if "{" in clean_str:
        start = clean_str.find("{")
        end = clean_str.rfind("}") + 1
        clean_str = clean_str[start:end]
    response_json = json.loads(clean_str)
    responses_refined.append(response_json)

df_refined_inter = pd.DataFrame(responses_refined)
df_refined_inter

,veracity_label,explanation_text,factor_scores
0,Pants on Fire,This article is demonstrably false and satiric...,"[{'factor': 'Authenticity', 'score': 1, 'reaso..."
1,Mostly True,The article reports on Walmart's financial per...,"[{'factor': 'Authenticity', 'score': 8, 'reaso..."
2,True,This article is a humorous piece of fictional ...,"[{'factor': 'Authenticity', 'score': 7, 'reaso..."
3,Mostly True,The article reports on upcoming studies from N...,"[{'factor': 'Authenticity', 'score': 9, 'reaso..."
4,True,This article presents factual information abou...,"[{'factor': 'Authenticity', 'score': 9, 'reaso..."
5,Mostly True,The article primarily reports on a shift in Pr...,"[{'factor': 'Authenticity', 'score': 8, 'reaso..."
6,Mostly True,The article reports a direct quote from Presid...,"[{'factor': 'Authenticity', 'score': 8, 'reaso..."
7,Mostly True,The article reports on trends in the San Diego...,"[{'factor': 'Authenticity', 'score': 8, 'reaso..."
8,Mostly True,The article reports on allegations against a C...,"[{'factor': 'Authenticity', 'score': 8, 'reaso..."
9,Mostly True,The article accurately reports on concerns rai...,"[{'factor': 'Authenticity', 'score': 8, 'reaso..."


In [8]:
df_list = []
for i in df_refined_inter['factor_scores']:
    df_temp = pd.DataFrame(i,)
    scores_wide = (
    df_temp
        .assign(_idx=0)
        .pivot(index="_idx", columns="factor", values="score")
        .reset_index()
        .drop(columns="_idx")
    )
    df_list.append(scores_wide)
df_refined = pd.concat(df_list)
df_refined.index = range(len(df_list))
df_refined

factor,Authenticity,Confirmation Bias,Political Bias,Sensationalism,Short-term Utility,Spam
0,1,2,7,10,8,2
1,8,2,3,2,3,1
2,7,1,1,3,2,1
3,9,2,1,2,1,1
4,9,1,1,2,3,1
5,8,3,5,4,2,1
6,8,3,7,6,2,1
7,8,2,1,2,3,1
8,8,3,5,4,2,1
9,8,4,6,3,2,1


## Final Prompt: In-context learning

In [9]:
responses_final = []
for i in range(1,data.shape[0]+1):
    txt = data.loc[i, 'text']
    response = run_genai(article_text=txt, prompt=final_prompt)
    clean_str = re.sub(r"```json|```", "", response).strip()
    if "{" in clean_str:
        start = clean_str.find("{")
        end = clean_str.rfind("}") + 1
        clean_str = clean_str[start:end]
    response_json = json.loads(clean_str)
    responses_final.append(response_json)

df_final_inter = pd.DataFrame(responses_final)
df_final_inter

,veracity_label,explanation_text,factor_scores
0,Pants on Fire,The article presents a highly improbable and s...,"[{'factor': 'Authenticity', 'score': 1, 'reaso..."
1,Mostly True,The article reports on Walmart's financial per...,"[{'factor': 'Authenticity', 'score': 8, 'reaso..."
2,True,The article reports a peculiar but verifiable ...,"[{'factor': 'Authenticity', 'score': 7, 'reaso..."
3,Mostly True,The article reports on upcoming clinical trial...,"[{'factor': 'Authenticity', 'score': 8, 'reaso..."
4,True,The article presents factual information about...,"[{'factor': 'Authenticity', 'score': 9, 'reaso..."
5,Mostly True,The article accurately reports on a shift in P...,"[{'factor': 'Authenticity', 'score': 8, 'reaso..."
6,Mostly False,The article reports on a direct call by Presid...,"[{'factor': 'Authenticity', 'score': 7, 'reaso..."
7,True,The article presents factual data regarding th...,"[{'factor': 'Authenticity', 'score': 9, 'reaso..."
8,Mostly True,The article reports on ongoing legal allegatio...,"[{'factor': 'Authenticity', 'score': 8, 'reaso..."
9,Mostly True,The article accurately reports on concerns rai...,"[{'factor': 'Authenticity', 'score': 8, 'reaso..."


In [10]:
df_list = []
for json_list in df_final_inter['factor_scores']:
    df_temp = pd.DataFrame(json_list,)
    scores_wide = (
    df_temp
        .assign(_idx=0)
        .pivot(index="_idx", columns="factor", values="score")
        .reset_index()
        .drop(columns="_idx")
    )
    df_list.append(scores_wide)
df_final = pd.concat(df_list)
df_final.index = range(len(df_list))
df_final

factor,Authenticity,Confirmation Bias,Political Bias,Sensationalism,Short-term Utility,Spam
0,1,6,7,10,8,9
1,8,4,2,3,3,1
2,7,1,1,4,5,1
3,8,2,1,3,3,1
4,9,1,1,2,3,1
5,8,3,3,4,3,1
6,7,3,6,6,4,2
7,9,2,1,2,3,1
8,8,3,2,3,3,1
9,8,4,6,3,3,1


## Fractal Chain of Thought

In [11]:
responses_fcot = []
for i in range(1,data.shape[0]+1):
    txt = data.loc[i, 'text']
    print(txt)
    response = run_genai(article_text=txt, prompt=fcot_prompt3_gpt)
    clean_str = re.sub(r"```json|```", "", response).strip()
    print(response)
    if "{" in clean_str:
        start = clean_str.find("{")
        end = clean_str.rfind("}") + 1
        clean_str = clean_str[start:end]
    response_json = json.loads(clean_str)
    responses_fcot.append(response_json)

df_fcot_inter = pd.DataFrame(responses_fcot)
df_fcot_inter

NEW YORK, NY — Mayor-elect Zohran Mamdani announced plans on Thursday to specifically target the rich with a new tax on anyone who makes more than $0.00 a year.

The plan, dubbed the "How Much You Got?" Tax, was expected to go into effect in January after Mamdani officially takes office. He has called the tax a day-one initiative, claiming his bread lines would be a disaster without the increased revenue from taking all of everyone's money.

"Everything will be free because we'll use everyone's money to pay for it," Mamdani said. "It's basic economics. You wouldn't understand."

Officially, Mamdani has no authority to raise taxes in the city (other than property taxes) without the cooperation of the New York State Legislature. However, experts believe he'll receive little pushback at the state level.

"He kind of just does whatever he wants without any logic or reason behind it, but I can't tell him not to because I'm not Islamophobic," New York Governor Kathy Hochul conceded.

For the

,veracity_label,explanation_text,factor_scores
0,Pants on Fire,This article is demonstrably false and relies ...,"[{'factor': 'Authenticity', 'score': 1, 'reaso..."
1,Mostly True,The article accurately reports on Walmart's re...,"[{'factor': 'Authenticity', 'score': 8, 'reaso..."
2,True,This article is a humorous piece of fictional ...,"[{'factor': 'Authenticity', 'score': 7, 'reaso..."
3,Mostly True,The article reports on upcoming studies from N...,"[{'factor': 'Authenticity', 'score': 8, 'reaso..."
4,True,This article presents factual information abou...,"[{'factor': 'Authenticity', 'score': 9, 'reaso..."
5,Mostly True,The article accurately reports on a shift in P...,"[{'factor': 'Authenticity', 'score': 8, 'reaso..."
6,Mostly True,The article reports a direct quote from Presid...,"[{'factor': 'Authenticity', 'score': 9, 'reaso..."
7,Mostly True,The article reports on trends in the San Diego...,"[{'factor': 'Authenticity', 'score': 9, 'reaso..."
8,Mostly True,The article reports on allegations against a C...,"[{'factor': 'Authenticity', 'score': 8, 'reaso..."
9,Mostly True,The article reports on concerns from US lawmak...,"[{'factor': 'Authenticity', 'score': 8, 'reaso..."


In [12]:
df_list = []
for json_list in df_fcot_inter['factor_scores']:
    df_temp = pd.DataFrame(json_list,)
    scores_wide = (
    df_temp
        .assign(_idx=0)
        .pivot(index="_idx", columns="factor", values="score")
        .reset_index()
        .drop(columns="_idx")
    )
    df_list.append(scores_wide)
df_final_fcot = pd.concat(df_list)
df_final_fcot.index = range(15)
df_final_fcot

factor,Authenticity,Confirmation Bias,Political Bias,Sensationalism,Short-term Utility,Spam
0,1,7,8,10,9,2
1,8,3,2,3,2,1
2,7,1,1,3,2,1
3,8,2,1,3,2,1
4,9,1,1,2,3,1
5,8,3,5,4,2,1
6,9,3,7,6,6,2
7,9,2,1,2,3,1
8,8,3,4,3,2,1
9,8,4,6,3,2,1


# Post-processing LLM Outputs

In [13]:
naive_df.index.name = 'id'
naive_df.columns.name = None
naive_df.rename(columns={'Short-term Utility':'Short-Term Utility'}, inplace=True)
# naive_df.index += 1
naive_df = naive_df[['Authenticity','Sensationalism','Political Bias','Spam','Confirmation Bias','Short-Term Utility']]
display(naive_df)

df_refined.index.name = 'id'
df_refined.columns.name = None
df_refined.rename(columns={'Short-term Utility':'Short-Term Utility'}, inplace=True)
# df_refined.index += 1
df_refined = df_refined[['Authenticity','Sensationalism','Political Bias','Spam','Confirmation Bias','Short-Term Utility']]
display(df_refined)

df_final.index.name = 'id'
df_final.columns.name = None
df_final.rename(columns={'Short-term Utility':'Short-Term Utility'}, inplace=True)
# df_final.index += 1
df_final = df_final[['Authenticity','Sensationalism','Political Bias','Spam','Confirmation Bias','Short-Term Utility']]
display(df_final)

,Authenticity,Sensationalism,Political Bias,Spam,Confirmation Bias,Short-Term Utility
id,,,,,,
0,1,9,7,2,4,3
1,8,3,4,1,5,7
2,8,6,1,1,2,4
3,9,3,2,1,4,7
4,10,2,1,1,3,8
5,8,4,6,1,5,7
6,9,7,8,1,6,8
7,9,2,3,1,4,8
8,9,5,6,1,5,8


,Authenticity,Sensationalism,Political Bias,Spam,Confirmation Bias,Short-Term Utility
id,,,,,,
0,1,10,7,2,2,8
1,8,2,3,1,2,3
2,7,3,1,1,1,2
3,9,2,1,1,2,1
4,9,2,1,1,1,3
5,8,4,5,1,3,2
6,8,6,7,1,3,2
7,8,2,1,1,2,3
8,8,4,5,1,3,2


,Authenticity,Sensationalism,Political Bias,Spam,Confirmation Bias,Short-Term Utility
id,,,,,,
0,1,10,7,9,6,8
1,8,3,2,1,4,3
2,7,4,1,1,1,5
3,8,3,1,1,2,3
4,9,2,1,1,1,3
5,8,4,3,1,3,3
6,7,6,6,2,3,4
7,9,2,1,1,2,3
8,8,3,2,1,3,3


In [14]:
df_final_fcot.index.name = 'id'
df_final_fcot.columns.name = None
df_final_fcot.rename(columns={'Short-term Utility':'Short-Term Utility'}, inplace=True)
# df_final_fcot.index += 1
df_final_fcot = df_final_fcot[['Authenticity','Sensationalism','Political Bias','Spam','Confirmation Bias','Short-Term Utility']]
display(df_final_fcot)

,Authenticity,Sensationalism,Political Bias,Spam,Confirmation Bias,Short-Term Utility
id,,,,,,
0,1,10,8,2,7,9
1,8,3,2,1,3,2
2,7,3,1,1,1,2
3,8,3,1,1,2,2
4,9,2,1,1,1,3
5,8,4,5,1,3,2
6,9,6,7,2,3,6
7,9,2,1,1,2,3
8,8,3,4,1,3,2


In [15]:
ground_truth = data.drop(columns=['text','Unnamed: 0'])
ground_truth.rename(columns={'short_term_utility_score':'Short-term Utility','toxicity_score':'Spam'}, inplace=True)
ground_truth.rename(columns=lambda x: x.replace('_score','').replace('_',' ').title(), inplace=True)
ground_truth.index -= 1
ground_truth

,Authenticity,Sensationalism,Political Bias,Spam,Confirmation Bias,Short-Term Utility
id,,,,,,
0,2,7,5,9,3,8
1,7,5,2,1,5,3
2,1,9,2,5,8,10
3,9,4,1,2,6,2
4,8,2,1,1,6,5
5,7,6,8,4,2,6
6,9,7,1,8,5,8
7,7,2,1,1,6,4
8,8,7,7,6,8,3


# Prompt Results

In [16]:
naive_results = ((naive_df == ground_truth)|(np.abs(naive_df-ground_truth)==1)).mean()
naive_results

Authenticity          0.800000
Sensationalism        0.466667
Political Bias        0.400000
Spam                  0.600000
Confirmation Bias     0.466667
Short-Term Utility    0.400000
dtype: float64

In [17]:
refined_results = ((df_refined == ground_truth)|(np.abs(df_refined-ground_truth)==1)).mean()
refined_results

Authenticity          0.933333
Sensationalism        0.333333
Political Bias        0.400000
Spam                  0.600000
Confirmation Bias     0.333333
Short-Term Utility    0.333333
dtype: float64

In [18]:
final_results = ((df_final == ground_truth)|(np.abs(df_final-ground_truth)==1)).mean()
final_results

Authenticity          0.800000
Sensationalism        0.400000
Political Bias        0.466667
Spam                  0.666667
Confirmation Bias     0.333333
Short-Term Utility    0.333333
dtype: float64

In [19]:
fcot_results = ((df_final_fcot == ground_truth)|(np.abs(df_final_fcot-ground_truth)==1)).mean()
fcot_results

Authenticity          0.866667
Sensationalism        0.400000
Political Bias        0.466667
Spam                  0.600000
Confirmation Bias     0.266667
Short-Term Utility    0.333333
dtype: float64

In [21]:
result_df = pd.concat([naive_results,refined_results,final_results, fcot_results], axis=1).T
result_df['Prompt'] = ['Naive','Refined','Final','FCOT']
result_df.set_index('Prompt', inplace=True)
result_df

,Authenticity,Sensationalism,Political Bias,Spam,Confirmation Bias,Short-Term Utility
Prompt,,,,,,
Naive,0.800000,0.466667,0.400000,0.600000,0.466667,0.400000
Refined,0.933333,0.333333,0.400000,0.600000,0.333333,0.333333
Final,0.800000,0.400000,0.466667,0.666667,0.333333,0.333333
FCOT,0.866667,0.400000,0.466667,0.600000,0.266667,0.333333


In [ ]:
result_df.to_csv('../data/prompting_results.csv')

In [ ]:
result_df = pd.read_csv('../data/prompting_results.csv')
result_df

,Prompt,Authenticity,Sensationalism,Political Bias,Spam,Confirmation Bias,Short-Term Utility
0,Naive,0.7,0.5,0.5,0.4,0.2,0.1
1,Refined,0.8,0.3,0.5,0.4,0.3,0.4
2,Final,0.7,0.4,0.5,0.5,0.3,0.6


In [ ]:
descs = np.array(["Standard prompt for our task at hand, giving minimal details for the LLM to work with",
         "Standard prompt + scoring recipes for the LLM to use and evaluate",
         "Standard prompt + scoring recipes and example outputs (in-context learning) for best possible results"])

prompts = np.array([baseline_prompt('<article>'),
                    refined_prompt('<article>','<predictive model output>'),
                    final_prompt('<article>','<predictive model output>')])

In [ ]:
result_df['description'] = descs
result_df['full_prompt'] = prompts

result_df

,Prompt,Authenticity,Sensationalism,Political Bias,Spam,Confirmation Bias,Short-Term Utility,description,full_prompt
0,Naive,0.7,0.5,0.5,0.4,0.2,0.1,"Standard prompt for our task at hand, giving m...",\nYou are an AI tasked with scoring news artic...
1,Refined,0.8,0.3,0.5,0.4,0.3,0.4,Standard prompt + scoring recipes for the LLM ...,\nYou are an AI assistant assigned to evaluate...
2,Final,0.7,0.4,0.5,0.5,0.3,0.6,Standard prompt + scoring recipes and example ...,You are an AI assistant assigned to evaluate t...


In [ ]:
result_df.to_csv('../data/prompting_results.csv')